Design, Sampling, and Collection
================================

**Author:** Ethan Ligon

**Date:** \today



What a careful analyst should know before touching the data.



## Reading



-   Deaton ch. 1 — the whole chapter
-   Deaton §1.4 on survey design and §2.2 on what to do about it
-   Skim the GLSS7 *Report* methodology section



## First contact with the data



Everything in the workshop starts the same way.



In [1]:
import lsms_library as ll

ghana = ll.Country('GhanaLSS')
print(ghana.waves)

Seven rounds, 1987–88 through 2016–17.  Ask what tables have been
harmonized for this country:



In [1]:
print(ghana.data_scheme)

Each name is also a method.  Calling it returns a `DataFrame` stacked across
every wave, indexed by some subset of $(t, v, i, \mathit{pid}, j)$ —
wave, cluster, household, person, item.



## The design, in the data



The `sample` table is where the design lives.



In [1]:
sample = ghana.sample()
sample.head()

The columns are the design: `v` is the enumeration area, `strata` the
stratum, `weight` the (nonresponse-adjusted) design weight, `Rural` the
urban/rural classification.  `panel_weight` applies only to the households
that recur across rounds.



In [1]:
import pandas as pd

# How many households per round, and how many EAs?
by_wave = sample.groupby('t').agg(
    households=('weight', 'size'),
    clusters=('v', 'nunique'),
    population=('weight', 'sum'),
)
by_wave['hh_per_cluster'] = by_wave.households / by_wave.clusters
by_wave

Two things to notice, and the second is a trap.

`hh_per_cluster` is the $m$ in the design-effect formula, and it sits
between twelve and twenty in every round — as designed.

`population` is the sum of the weights, and it comes out *exactly* equal to
the number of households in every round.  So these weights are not expansion
factors: they have been normalized to average one, which is the usual
convention in harmonized data.  That normalization is harmless for any
weighted mean or share, because the constant cancels in the ratio.  It is
fatal if you want a total — the number of poor people in Ghana, say — for
which you need the original expansion factors from the survey documentation.
Check what your weights are normalized to before you compute a total.



## Weighted versus unweighted



Take the simplest possible descriptive statistic — the rural share — and
compute it both ways.



In [1]:
s = sample.dropna(subset=['weight', 'Rural'])
rural = (s.Rural.str.strip().str.lower() == 'rural').astype(float)

unweighted = rural.groupby(s.index.get_level_values('t')).mean()
weighted = (rural * s.weight).groupby(s.index.get_level_values('t')).sum() \
           / s.weight.groupby(s.index.get_level_values('t')).sum()

pd.DataFrame({'unweighted': unweighted, 'weighted': weighted}).round(3)

Look at the two columns carefully.

For the four earliest rounds they agree to three decimals — because in
those rounds every household carries weight exactly one, so there is no
weight variation to apply.  From 2005–06 the columns separate, and by
2016–17 they differ by thirteen percentage points: 0.57 unweighted against
0.44 weighted.  That is not noise, and it is not small.  It is the design,
and only the second column is an estimate of the rural share of Ghana.

A result that moves by thirteen points depending on a keyword argument is
worth remembering the next time weighting looks like a technicality.



## Estimating the design effect



The design effect is not a property of the survey; it is a property of the
survey *and the variable*.  Estimate it directly, by comparing the clustered
variance of a mean to the simple-random-sample variance.



In [1]:
import numpy as np

def design_effect(y, cluster):
    """deff for the mean of y, clusters given by `cluster`."""
    # float64, not the arrow-backed bool pandas 3 hands back: pyarrow
    # booleans have no .var().
    df = pd.DataFrame({'y': pd.Series(y).astype('float64'),
                       'g': pd.Series(cluster).astype('object')}).dropna()
    n, ybar = len(df), df.y.mean()
    v_srs = df.y.var(ddof=1) / n
    # between-cluster variance of the cluster totals (Huber-White, one-way)
    tot = df.groupby('g').y.apply(lambda s: (s - ybar).sum())
    v_cl = (tot ** 2).sum() / n ** 2
    return v_cl / v_srs

wave = '2016-17'
w = s.xs(wave, level='t')
print(f"m   = {len(w) / w.v.nunique():.1f} households per EA")
print(f"deff= {design_effect(w.Rural.str.strip().str.lower().eq('rural'), w.v):.2f}")

Rural status is close to constant within an enumeration area, so $\rho$ is
near one and the design effect is close to $m$.  That is the extreme case.
Try it on something less spatially clustered — household size, say — and
watch the number fall.



## Exercises



1.  Repeat the weighted/unweighted comparison for mean household size.  Does
    the gap have the same sign as it did for the rural share?  Explain why.
2.  Compute `deff` for household size in 2016–17, and back out the implied
    $\rho$.  How many households would a simple random sample need in order
    to match the precision of this survey?
3.  The weights sum to the sample size, so they cannot give you a population
    total.  Find the expansion factors in the GLSS documentation and work out
    what constant each round's weights were divided by.  Which quantities
    computed above would change if you used the raw factors instead, and
    which are invariant?

